# Kirana — training dan evaluasi kuis
Satu notebook untuk sesi lama dan sesi berkategori. Sesi lama menggunakan model yang sudah diekspor jika tersedia; kategori baru dilatih dengan simulator dan pemilih soal backend. Evaluasi hanya pada data sintetis, bukan bukti akurasi siswa nyata.
Ekspor nonaktif secara bawaan. Jalankan sel berurutan dengan lingkungan Python yang sama seperti Flask.

In [7]:
from pathlib import Path
import json, sys
root = next((p for parent in (Path.cwd(), *Path.cwd().parents) for p in (parent, parent / 'Source Code' / 'Kirana') if (p / 'ml' / 'unified_training.py').is_file()), None)
if root is None: raise FileNotFoundError('Buka notebook dari folder proyek Kirana.')
sys.path.insert(0, str(root))
from ml.unified_training import prepare_legacy, train_categories, export_and_verify
from ml.training import input_checksums
from ml.paths import ARTIFACT, DATA_DIR, ML_DIR
from ml.model_bundle import DEFAULT_PATH, load_bundle
from backend.src.quiz import read_bank, bank_coverage
SEED = 42
EXPORT_MODEL = True
print('Model lama:', ARTIFACT if ARTIFACT.exists() else 'tidak ada; akan dilatih ulang')
print('Tujuan ekspor:', DEFAULT_PATH)

Model lama: C:\Davyd\Kirana\Source Code\Kirana\ml\artifacts\kirana_mcq_v1.joblib
Tujuan ekspor: C:\Davyd\Kirana\Source Code\Kirana\ml\artifacts\kirana_model.joblib


## Audit bank dan input
Periksa keluarga soal per operasi, kategori, dan split sebelum melatih. CSV respons asli tidak diubah.

In [2]:
coverage = bank_coverage(read_bank(DATA_DIR))
for row in coverage: print(row)
print('Hash input:', json.dumps(input_checksums(DATA_DIR), indent=2))

{'split': 'train', 'operation': 'addition', 'category': 'tens', 'families': 205, 'shown_opportunities': {'A1': 135, 'A2': 92, 'A3': 0, 'A4': 131}}
{'split': 'train', 'operation': 'addition', 'category': 'hundreds', 'families': 280, 'shown_opportunities': {'A1': 156, 'A2': 117, 'A3': 72, 'A4': 148}}
{'split': 'train', 'operation': 'addition', 'category': 'thousands', 'families': 210, 'shown_opportunities': {'A1': 111, 'A2': 97, 'A3': 62, 'A4': 115}}
{'split': 'train', 'operation': 'subtraction', 'category': 'tens', 'families': 210, 'shown_opportunities': {'S1': 104, 'S2': 104, 'S3': 22, 'S4': 22, 'S5': 0, 'S6': 0}}
{'split': 'train', 'operation': 'subtraction', 'category': 'hundreds', 'families': 280, 'shown_opportunities': {'S1': 178, 'S2': 182, 'S3': 51, 'S4': 30, 'S5': 50, 'S6': 94}}
{'split': 'train', 'operation': 'subtraction', 'category': 'thousands', 'families': 210, 'shown_opportunities': {'S1': 121, 'S2': 130, 'S3': 38, 'S4': 24, 'S5': 39, 'S6': 108}}
{'split': 'validation', 'o

## Pertahankan sesi lama
Jika artefak v1 ada, gunakan estimator dan ambangnya persis seperti sebelumnya. Jika tidak ada, latih dari split lama dan tampilkan evaluasi test. Jangan menghapus artefak lama sebelum hasil baru diperiksa.

In [3]:
legacy_models, legacy_info = prepare_legacy(SEED)
print(json.dumps(legacy_info, indent=2))

{
  "source": "existing_legacy_artifact",
  "sha256": "4e5815be4d1e1c42d7182b195c6867590028108d81a4bf58ba46da6d9c49df03"
}


## Simulasi, training, dan evaluasi kategori
Untuk setiap operasi, model multilabel dilatih pada train lintas kategori. Ambang dipilih pada validation per kategori, lalu test hanya dievaluasi. Setiap percobaan berisi 9 soal inti dan 6 adaptif.

In [4]:
category_models, report, samples = train_categories(SEED)
report['legacy'] = legacy_info
print(json.dumps(report['results'], indent=2, ensure_ascii=False))

{
  "addition": {
    "tens": {
      "thresholds": {
        "rules": [
          0.36,
          0.36,
          1.0000000000000002,
          0.42
        ],
        "logistic": [
          0.85,
          0.92,
          1.0000000000000002,
          0.92
        ]
      },
      "validation": {
        "rules": {
          "sessions": 1320,
          "macro_f1": 0.8708947916334814,
          "false_pattern_calls": 17,
          "true_patterns_found": 779,
          "true_patterns": 992,
          "mixed_sessions": 152,
          "mixed_both_found": 24,
          "no_target_sessions": 480,
          "no_target_with_calls": 17,
          "abstained": 548,
          "preliminary_false_calls": 316,
          "by_pattern": {
            "A1": {
              "precision": 0.9822064056939501,
              "recall": 0.8363636363636363,
              "f1": 0.9034369885433715,
              "support": 330
            },
            "A2": {
              "precision": 0.969811320754717,
    

## Simpan laporan evaluasi
Tinjau precision, recall, dugaan keliru, pola kedua MIXED, dan jumlah sesi tanpa dugaan. Hasil sintetis tidak menggantikan uji pada siswa nyata.

In [5]:
report_path = ML_DIR / 'reports' / 'category_evaluation.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Laporan:', report_path)

Laporan: C:\Davyd\Kirana\Source Code\Kirana\ml\reports\category_evaluation.json


## Ekspor satu model untuk Flask
Setelah meninjau hasil, ubah EXPORT_MODEL menjadi True di sel persiapan, jalankan ulang sel itu lalu sel ini. Ekspor memuat ulang model dan membandingkan skor serta ambang. Restart Flask setelah berhasil.

In [8]:
if EXPORT_MODEL:
    path = export_and_verify(legacy_models, category_models, samples, legacy_info)
    bundle = load_bundle(path)
    print('Model gabungan:', path, 'cabang:', ['legacy', 'category'])
else:
    print('Ekspor nonaktif; model lama tetap digunakan Flask.')

Model gabungan: C:\Davyd\Kirana\Source Code\Kirana\ml\artifacts\kirana_model.joblib cabang: ['legacy', 'category']
